# 📊 Calidad y Limpieza de Datos: Guía Práctica Completa

## Big Data Aplicado - Máster en Big Data e Inteligencia Artificial

---

**Versión:** 1.0  
**Fecha:** Noviembre 2025  
**Duración estimada:** 6-8 horas

---

## 🎯 Objetivos del Notebook

Este notebook proporciona una comprensión práctica y profunda de los procesos de **limpieza y calidad de datos**, elementos fundamentales en cualquier proyecto de Big Data e Inteligencia Artificial.

### 📚 Contenido:

1. **📦 Integración de Datos (2.1)** - Fusión horizontal y vertical con normalización
2. **🔍 Selección y Reducción (2.2)** - PCA y técnicas de sampling
3. **🔄 Conversión y Transformación (2.3)** - Normalización, Box-Cox, discretización
4. **⚠️ Valores Extremos (2.5)** - Detección con 3σ, IQR y Mahalanobis
5. **📈 Correlaciones (3.5)** - Pearson vs Spearman
6. **❌ Malas Prácticas** - Impacto de NO aplicar calidad de datos

---

### 💡 Dato Importante

**Se estima que el 80% del trabajo de un científico de datos se invierte en procesos de limpieza.**

### 💰 Impacto Económico

**Coste de validar y corregir errores:**
- **A la entrada:** 1€ por error
- **Después de la ingesta:** Entre 10€ y 100€ por error

**ROI de la calidad de datos:** 1,000% - 10,000%

---

## 📦 1. Instalación de Librerías

Instalamos todas las librerías necesarias para el análisis.

In [ ]:
# Instalación de librerías necesarias
!pip install numpy pandas matplotlib seaborn scikit-learn scipy missingno -q

print("✅ Todas las librerías han sido instaladas correctamente")

## 📚 2. Importación de Librerías y Configuración

In [ ]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from scipy.stats import boxcox

# Scikit-learn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, silhouette_score

# Configuración de visualización
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")
print(f"   - NumPy versión: {np.__version__}")
print(f"   - Pandas versión: {pd.__version__}")

---

## 🔗 2.1 Integración de Datos

### Introducción

La **integración o fusión de datos** consiste en la combinación de datos procedentes de múltiples fuentes, con el fin de crear una estructura de datos coherente y única que contenga mayor cantidad de información.

**Tipos de fusión:**
- **Horizontal:** Añadir nuevos atributos (columnas)
- **Vertical:** Añadir nuevos registros (filas)

**Desafíos comunes:**
- Diferentes formatos y unidades
- Inconsistencias en nombres
- Duplicados
- Valores faltantes

### 2.1.1 Generación de Datos Sintéticos

Vamos a simular datos de tres tiendas de una cadena internacional con diferentes formatos.

In [ ]:
# Simulación de datos de diferentes tiendas de una cadena

# Tienda Madrid (Europa) - Datos en sistema métrico
tienda_madrid = pd.DataFrame({
    'producto_id': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'nombre': ['Laptop Pro', 'Mouse Inalámbrico', 'Teclado Mecánico', 'Monitor 27"', 'Webcam HD'],
    'precio_eur': [1200, 25, 89, 350, 65],
    'peso_kg': [2.5, 0.1, 0.8, 5.5, 0.3],
    'stock': [15, 50, 30, 20, 40],
    'tienda': 'Madrid'
})

# Tienda Nueva York (USA) - Datos en sistema imperial
tienda_nueva_york = pd.DataFrame({
    'product_id': ['P001', 'P002', 'P006', 'P007', 'P008'],
    'name': ['Laptop Pro', 'Wireless Mouse', 'USB Hub', 'External SSD 1TB', 'Headphones'],
    'price_usd': [1300, 28, 35, 120, 85],
    'weight_lb': [5.5, 0.22, 0.3, 0.5, 0.6],
    'inventory': [12, 45, 60, 25, 35],
    'store': 'New York'
})

# Tienda Barcelona (Europa) - Con inconsistencias
tienda_barcelona = pd.DataFrame({
    'producto_id': ['P003', 'P004', 'P009', 'P010', 'P001'],
    'nombre': ['Teclado Mecánico', 'Monitor 27"', 'Cable HDMI 2m', 'Hub USB-C', 'laptop pro'],
    'precio_eur': [92, 355, 15, 45, 1250],
    'peso_kg': [0.85, 5.6, 0.2, 0.15, 2.6],
    'stock': [25, 18, 100, 50, 10],
    'tienda': 'Barcelona'
})

print("📊 DATOS GENERADOS DE MÚLTIPLES TIENDAS")
print("\n" + "="*80)
print("🏪 TIENDA MADRID (Sistema Métrico)")
print("="*80)
display(tienda_madrid)

print("\n" + "="*80)
print("🏪 TIENDA NUEVA YORK (Sistema Imperial)")
print("="*80)
display(tienda_nueva_york)

print("\n" + "="*80)
print("🏪 TIENDA BARCELONA (Con Inconsistencias)")
print("="*80)
display(tienda_barcelona)

### 2.1.2 Fusión Vertical: Consolidando Inventarios

Proceso completo de integración con normalización, conversión de unidades y detección de duplicados.

In [ ]:
# PASO 1: Normalizar nombres de columnas
print("🔧 PASO 1: Normalización de Nombres de Columnas")
print("="*80)

# Renombrar columnas de Nueva York al formato europeo
tienda_nueva_york_norm = tienda_nueva_york.rename(columns={
    'product_id': 'producto_id',
    'name': 'nombre',
    'price_usd': 'precio_original',
    'weight_lb': 'peso_original',
    'inventory': 'stock',
    'store': 'tienda'
})
tienda_nueva_york_norm['unidad_precio'] = 'USD'
tienda_nueva_york_norm['unidad_peso'] = 'lb'

# Renombrar columnas de Madrid y Barcelona
tienda_madrid_norm = tienda_madrid.rename(columns={
    'precio_eur': 'precio_original',
    'peso_kg': 'peso_original'
})
tienda_madrid_norm['unidad_precio'] = 'EUR'
tienda_madrid_norm['unidad_peso'] = 'kg'

tienda_barcelona_norm = tienda_barcelona.rename(columns={
    'precio_eur': 'precio_original',
    'peso_kg': 'peso_original'
})
tienda_barcelona_norm['unidad_precio'] = 'EUR'
tienda_barcelona_norm['unidad_peso'] = 'kg'

print("✅ Columnas normalizadas")

# PASO 2: Convertir unidades
print("\n🔧 PASO 2: Conversión de Unidades al Sistema Métrico")
print("="*80)

USD_TO_EUR = 0.92
LB_TO_KG = 0.453592

def convertir_a_metrico(df):
    df_conv = df.copy()
    df_conv['precio_eur'] = df_conv.apply(
        lambda row: row['precio_original'] * USD_TO_EUR if row['unidad_precio'] == 'USD' else row['precio_original'],
        axis=1
    )
    df_conv['peso_kg'] = df_conv.apply(
        lambda row: row['peso_original'] * LB_TO_KG if row['unidad_peso'] == 'lb' else row['peso_original'],
        axis=1
    )
    return df_conv

tienda_madrid_final = convertir_a_metrico(tienda_madrid_norm)
tienda_nueva_york_final = convertir_a_metrico(tienda_nueva_york_norm)
tienda_barcelona_final = convertir_a_metrico(tienda_barcelona_norm)

print(f"   Tasa USD->EUR: {USD_TO_EUR}")
print(f"   Tasa lb->kg: {LB_TO_KG}")
print("✅ Unidades convertidas")

# PASO 3: Normalizar nombres de productos
print("\n🔧 PASO 3: Normalización de Nombres")
print("="*80)

def normalizar_nombre(nombre):
    return nombre.strip().title()

tienda_madrid_final['nombre'] = tienda_madrid_final['nombre'].apply(normalizar_nombre)
tienda_nueva_york_final['nombre'] = tienda_nueva_york_final['nombre'].apply(normalizar_nombre)
tienda_barcelona_final['nombre'] = tienda_barcelona_final['nombre'].apply(normalizar_nombre)

print("✅ Nombres normalizados")

# PASO 4: Fusión vertical
print("\n🔧 PASO 4: Fusión Vertical")
print("="*80)

columnas_finales = ['producto_id', 'nombre', 'precio_eur', 'peso_kg', 'stock', 'tienda']

inventario_consolidado = pd.concat([
    tienda_madrid_final[columnas_finales],
    tienda_nueva_york_final[columnas_finales],
    tienda_barcelona_final[columnas_finales]
], ignore_index=True)

print("\n📊 INVENTARIO CONSOLIDADO:")
display(inventario_consolidado)

print(f"\n📈 Estadísticas:")
print(f"   Total registros: {len(inventario_consolidado)}")
print(f"   Productos únicos: {inventario_consolidado['producto_id'].nunique()}")
print(f"   Tiendas: {inventario_consolidado['tienda'].nunique()}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stock_por_tienda = inventario_consolidado.groupby('tienda')['stock'].sum().sort_values()
stock_por_tienda.plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_title('Stock Total por Tienda', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Stock Total')

precio_medio = inventario_consolidado.groupby('tienda')['precio_eur'].mean().sort_values()
precio_medio.plot(kind='barh', ax=axes[1], color='lightcoral')
axes[1].set_title('Precio Medio por Tienda (EUR)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Precio Medio')

plt.tight_layout()
plt.show()

print("\n✅ Integración completada con éxito")

---

## 🔍 2.2 Selección y Reducción de Datos

### Introducción

Trabajar con grandes cantidades de datos puede convertir la tarea de análisis en un proceso muy complejo. Las **técnicas de reducción de datos** permiten obtener una representación reducida del conjunto de datos, manteniendo la integridad de la muestra original.

**Tipos:**
- **Reducción de dimensionalidad:** Reducir atributos (PCA)
- **Reducción de cantidad:** Reducir registros (sampling)

### 2.2.1 Análisis de Componentes Principales (PCA)

El PCA permite describir un conjunto de datos de n atributos en términos de m nuevas variables no correlacionadas.

In [ ]:
# Generar dataset de automóviles
np.random.seed(42)
n_coches = 50

cilindrada = np.random.uniform(1.0, 6.0, n_coches)
potencia = 50 + cilindrada * 30 + np.random.normal(0, 15, n_coches)
consumo = 4 + cilindrada * 1.5 + np.random.normal(0, 0.8, n_coches)
peso = 800 + cilindrada * 200 + np.random.normal(0, 100, n_coches)
aceleracion = 15 - (potencia / peso) * 100 + np.random.normal(0, 1, n_coches)
precio = 15000 + potencia * 150 + np.random.normal(0, 3000, n_coches)
velocidad_max = 140 + potencia * 0.5 + np.random.normal(0, 10, n_coches)
num_cilindros = np.round(cilindrada * 1.2 + 1).astype(int)
num_marchas = np.random.choice([5, 6, 7, 8], n_coches, p=[0.3, 0.4, 0.2, 0.1])

coches_df = pd.DataFrame({
    'modelo': [f'Modelo_{i+1}' for i in range(n_coches)],
    'cilindrada_L': cilindrada,
    'potencia_CV': potencia,
    'consumo_L100km': consumo,
    'peso_kg': peso,
    'aceleracion_0_100': aceleracion,
    'precio_EUR': precio,
    'velocidad_max_kmh': velocidad_max,
    'num_cilindros': num_cilindros,
    'num_marchas': num_marchas
})

print("🚗 DATASET DE AUTOMÓVILES")
print(f"Dimensiones: {coches_df.shape[0]} coches x {coches_df.shape[1]} características")
display(coches_df.head(10))

# Matriz de correlación
variables_numericas = ['cilindrada_L', 'potencia_CV', 'consumo_L100km', 'peso_kg',
                       'aceleracion_0_100', 'precio_EUR', 'velocidad_max_kmh',
                       'num_cilindros', 'num_marchas']

correlacion = coches_df[variables_numericas].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación - Antes de PCA', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\n💡 Variables altamente correlacionadas indican redundancia")

In [ ]:
# Aplicar PCA
print("🔬 APLICACIÓN DE PCA")
print("="*80)

# Preparar datos
X = coches_df[variables_numericas].values
print(f"\nDatos originales: {X.shape[0]} muestras x {X.shape[1]} características")

# Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Datos estandarizados (media=0, std=1)")

# PCA completo
pca_full = PCA()
pca_full.fit(X_scaled)

varianza_explicada = pca_full.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

print("\n📊 Varianza explicada por componente:")
for i, (var, var_acum) in enumerate(zip(varianza_explicada, varianza_acumulada)):
    print(f"   PC{i+1}: {var*100:.2f}% (Acumulada: {var_acum*100:.2f}%)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Varianza por componente
axes[0].bar(range(1, len(varianza_explicada)+1), varianza_explicada * 100,
            alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Componente Principal', fontsize=11)
axes[0].set_ylabel('Varianza Explicada (%)', fontsize=11)
axes[0].set_title('Varianza por Componente', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Scree plot
axes[1].plot(range(1, len(varianza_acumulada)+1), varianza_acumulada * 100,
             marker='o', linestyle='-', linewidth=2, markersize=8, color='darkred')
axes[1].axhline(y=80, color='green', linestyle='--', label='80% varianza')
axes[1].axhline(y=90, color='orange', linestyle='--', label='90% varianza')
axes[1].set_xlabel('Número de Componentes', fontsize=11)
axes[1].set_ylabel('Varianza Acumulada (%)', fontsize=11)
axes[1].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Número óptimo
n_comp_80 = np.argmax(varianza_acumulada >= 0.80) + 1
n_comp_90 = np.argmax(varianza_acumulada >= 0.90) + 1

print(f"\n✅ Componentes necesarios:")
print(f"   80% varianza: {n_comp_80} componentes")
print(f"   90% varianza: {n_comp_90} componentes")
print(f"   Reducción: {len(variables_numericas)} → {n_comp_80} dimensiones ({(1-n_comp_80/len(variables_numericas))*100:.1f}% reducción)")

# Aplicar PCA con componentes óptimos
pca = PCA(n_components=n_comp_80)
X_pca = pca.fit_transform(X_scaled)

print(f"\n✅ PCA aplicado: {X.shape} → {X_pca.shape}")

---

## 🔄 2.3 Conversión y Transformación de Datos

### Introducción

En la etapa de **conversión**, los datos son transformados para que el análisis posterior sea más eficiente.

**Principales técnicas:**
- **Normalización:** Min-Max y Z-Score
- **Transformación Box-Cox:** Para lograr normalidad
- **Discretización:** Convertir numéricos en categóricos

In [ ]:
# Generar datos con diferentes escalas
np.random.seed(42)
n = 100

datos_escalas = pd.DataFrame({
    'edad': np.random.randint(18, 70, n),
    'salario_EUR': np.random.randint(20000, 100000, n),
    'horas_trabajo': np.random.randint(20, 60, n),
    'años_experiencia': np.random.randint(0, 30, n),
    'satisfaccion': np.random.uniform(1, 10, n)
})

print("📊 DATASET CON DIFERENTES ESCALAS")
display(datos_escalas.describe())

# Normalización Min-Max
scaler_minmax = MinMaxScaler()
datos_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(datos_escalas),
    columns=datos_escalas.columns
)

# Normalización Z-Score
scaler_zscore = StandardScaler()
datos_zscore = pd.DataFrame(
    scaler_zscore.fit_transform(datos_escalas),
    columns=datos_escalas.columns
)

# Comparación visual
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

var_ejemplo = 'salario_EUR'

axes[0].hist(datos_escalas[var_ejemplo], bins=20, color='lightcoral', alpha=0.7, edgecolor='black')
axes[0].set_title(f'Original\n[{datos_escalas[var_ejemplo].min():.0f}, {datos_escalas[var_ejemplo].max():.0f}]', fontweight='bold')

axes[1].hist(datos_minmax[var_ejemplo], bins=20, color='lightgreen', alpha=0.7, edgecolor='black')
axes[1].set_title('Min-Max\n[0, 1]', fontweight='bold')

axes[2].hist(datos_zscore[var_ejemplo], bins=20, color='skyblue', alpha=0.7, edgecolor='black')
axes[2].set_title('Z-Score\n(μ=0, σ=1)', fontweight='bold')

plt.suptitle(f'Comparación de Normalización - {var_ejemplo}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Min-Max: Valores en [0,1]")
print("✅ Z-Score: Media=0, Std=1")

---

## ⚠️ 2.5 Valores Extremos (Outliers)

### Introducción

Los **outliers** son datos que se encuentran muy alejados de la distribución normal. Pueden ser:
- ✅ Variabilidad natural (legítimos)
- ❌ Errores de medición o entrada

**Métodos de detección:**
1. 3 Desviaciones Estándar (3σ)
2. Rango Intercuartílico (IQR)
3. Distancia de Mahalanobis

In [ ]:
# Generar datos con outliers
np.random.seed(42)
n_normal = 200
n_outliers = 20

salarios_normales = np.random.normal(loc=45000, scale=8000, size=n_normal)
outliers_bajos = np.random.uniform(5000, 15000, 5)
outliers_altos = np.random.uniform(150000, 300000, 15)

todos_salarios = np.concatenate([salarios_normales, outliers_bajos, outliers_altos])

df_salarios = pd.DataFrame({
    'salario': todos_salarios,
    'tipo': ['Normal']*n_normal + ['Outlier Bajo']*5 + ['Outlier Alto']*15
})
df_salarios = df_salarios.sample(frac=1, random_state=42).reset_index(drop=True)

print("📊 DATASET CON OUTLIERS")
print(f"Total: {len(df_salarios)} registros")
print(f"Outliers añadidos: {n_outliers}")

# MÉTODO 1: 3 Desviaciones Estándar
media = df_salarios['salario'].mean()
std = df_salarios['salario'].std()
limite_inf = media - 3 * std
limite_sup = media + 3 * std

df_salarios['outlier_3std'] = ((df_salarios['salario'] < limite_inf) |
                                (df_salarios['salario'] > limite_sup))

print(f"\n🔍 MÉTODO 3σ:")
print(f"   Límite inferior: {limite_inf:,.2f}€")
print(f"   Límite superior: {limite_sup:,.2f}€")
print(f"   Outliers detectados: {df_salarios['outlier_3std'].sum()}")

# MÉTODO 2: IQR
Q1 = df_salarios['salario'].quantile(0.25)
Q3 = df_salarios['salario'].quantile(0.75)
IQR = Q3 - Q1
limite_inf_iqr = Q1 - 1.5 * IQR
limite_sup_iqr = Q3 + 1.5 * IQR

df_salarios['outlier_iqr'] = ((df_salarios['salario'] < limite_inf_iqr) |
                               (df_salarios['salario'] > limite_sup_iqr))

print(f"\n🔍 MÉTODO IQR:")
print(f"   Q1: {Q1:,.2f}€, Q3: {Q3:,.2f}€")
print(f"   IQR: {IQR:,.2f}€")
print(f"   Outliers detectados: {df_salarios['outlier_iqr'].sum()}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter 3σ
outliers_3std = df_salarios[df_salarios['outlier_3std']]
normales = df_salarios[~df_salarios['outlier_3std']]

axes[0].scatter(range(len(normales)), normales['salario'], color='blue', alpha=0.5, s=30)
axes[0].scatter(outliers_3std.index, outliers_3std['salario'], color='red', s=100, marker='X', alpha=0.8)
axes[0].axhline(y=limite_sup, color='orange', linestyle='--', linewidth=2)
axes[0].axhline(y=limite_inf, color='orange', linestyle='--', linewidth=2)
axes[0].set_title(f'Método 3σ\nDetectados: {len(outliers_3std)}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Salario (€)')
axes[0].grid(alpha=0.3)

# Boxplot IQR
bp = axes[1].boxplot(df_salarios['salario'], vert=True, widths=0.5, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7))
outliers_iqr = df_salarios[df_salarios['outlier_iqr']]
axes[1].scatter([1]*len(outliers_iqr), outliers_iqr['salario'],
               color='red', s=100, marker='D', alpha=0.6, zorder=3)
axes[1].set_title(f'Método IQR\nDetectados: {len(outliers_iqr)}', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Salario (€)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Detección de Outliers: Comparación de Métodos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Outliers detectados y visualizados")

---

## 📈 3.5 Análisis de Correlaciones

### Introducción

El **coeficiente de correlación** mide la asociación entre dos variables (valores entre -1 y 1).

**Tipos:**
- **Pearson:** Relación LINEAL, requiere normalidad
- **Spearman:** Relación MONÓTONA, no requiere normalidad

In [ ]:
# Generar diferentes tipos de correlaciones
np.random.seed(42)
n = 100
x = np.linspace(0, 10, n)

y_lineal_pos = 2 * x + 5 + np.random.normal(0, 2, n)
y_lineal_neg = -1.5 * x + 20 + np.random.normal(0, 1.5, n)
y_cuadratica = 0.5 * (x - 5)**2 + np.random.normal(0, 1, n)
y_sin_corr = np.random.normal(10, 3, n)

df_corr = pd.DataFrame({
    'x': x,
    'lineal_positiva': y_lineal_pos,
    'lineal_negativa': y_lineal_neg,
    'cuadratica': y_cuadratica,
    'sin_correlacion': y_sin_corr
})

print("📊 ANÁLISIS DE CORRELACIONES")
print("="*80)

# Calcular correlaciones
variables = ['lineal_positiva', 'lineal_negativa', 'cuadratica', 'sin_correlacion']
resultados = []

for var in variables:
    r_pearson, p_pearson = stats.pearsonr(df_corr['x'], df_corr[var])
    r_spearman, p_spearman = stats.spearmanr(df_corr['x'], df_corr[var])

    resultados.append({
        'Variable': var,
        'Pearson': f"{r_pearson:.3f}",
        'Spearman': f"{r_spearman:.3f}",
        'Diferencia': f"{abs(r_pearson - r_spearman):.3f}"
    })

    print(f"\n{var}:")
    print(f"   Pearson:  {r_pearson:.4f} (p={p_pearson:.4f})")
    print(f"   Spearman: {r_spearman:.4f} (p={p_spearman:.4f})")

df_resultados = pd.DataFrame(resultados)
print("\n📊 TABLA COMPARATIVA:")
display(df_resultados)

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

titulos = ['Lineal Positiva', 'Lineal Negativa', 'Cuadrática', 'Sin Correlación']

for idx, (var, titulo) in enumerate(zip(variables, titulos)):
    r_p, _ = stats.pearsonr(df_corr['x'], df_corr[var])
    r_s, _ = stats.spearmanr(df_corr['x'], df_corr[var])

    axes[idx].scatter(df_corr['x'], df_corr[var], alpha=0.6, s=50, edgecolors='black')
    axes[idx].set_title(f'{titulo}\nPearson: {r_p:.3f} | Spearman: {r_s:.3f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('X')
    axes[idx].set_ylabel('Y')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Comparación: Pearson vs Spearman', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 CONCLUSIÓN:")
print("   Pearson detecta relaciones LINEALES")
print("   Spearman detecta relaciones MONÓTONAS")
print("   Gran diferencia entre ambos indica NO LINEALIDAD")

---

## ❌ 6. MALAS PRÁCTICAS: Impacto de NO Aplicar Calidad de Datos

### Introducción

En esta sección demostraremos el **impacto crítico** de NO aplicar técnicas de calidad de datos. Compararemos los resultados de análisis y modelos con y sin limpieza.

**Escenarios:**
1. Análisis sin normalización
2. Modelos sin manejo de outliers
3. Correlaciones sin verificar normalidad

### 6.1 Escenario 1: Clustering Sin Normalización

**Problema:** Aplicar K-Means sin normalizar datos con diferentes escalas.

In [ ]:
# Generar datos de empleados con escalas MUY diferentes
np.random.seed(42)
n = 150

datos_empleados = pd.DataFrame({
    'edad': np.random.randint(22, 65, n),
    'salario_anual': np.random.randint(25000, 120000, n),
    'años_experiencia': np.random.randint(0, 40, n),
    'horas_formacion': np.random.randint(10, 200, n),
    'satisfaccion_0_10': np.random.uniform(3, 10, n)
})

print("📊 DATASET DE EMPLEADOS")
print(f"Registros: {len(datos_empleados)}")
print("\nRangos de variables:")
for col in datos_empleados.columns:
    print(f"   {col}: [{datos_empleados[col].min():.0f}, {datos_empleados[col].max():.0f}]")

X = datos_empleados.values

# ❌ MALA PRÁCTICA: K-Means SIN normalización
print("\n" + "="*80)
print("❌ MALA PRÁCTICA: K-Means SIN Normalización")
print("="*80)

kmeans_sin_norm = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_sin_norm = kmeans_sin_norm.fit_predict(X)
silhouette_sin_norm = silhouette_score(X, clusters_sin_norm)

print(f"\nSilhouette Score: {silhouette_sin_norm:.4f}")
print(f"Distribución: {np.bincount(clusters_sin_norm)}")
print("\n⚠️  PROBLEMA: El salario (25k-120k) DOMINA el clustering")
print("   Variables como satisfacción (3-10) tienen IMPACTO MÍNIMO")

# ✅ BUENA PRÁCTICA: K-Means CON normalización
print("\n" + "="*80)
print("✅ BUENA PRÁCTICA: K-Means CON Normalización")
print("="*80)

scaler_emp = StandardScaler()
X_norm = scaler_emp.fit_transform(X)

kmeans_con_norm = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_con_norm = kmeans_con_norm.fit_predict(X_norm)
silhouette_con_norm = silhouette_score(X_norm, clusters_con_norm)

print(f"\nSilhouette Score: {silhouette_con_norm:.4f}")
print(f"Distribución: {np.bincount(clusters_con_norm)}")
print(f"\n✅ MEJORA: {silhouette_sin_norm:.4f} → {silhouette_con_norm:.4f}")
print(f"   Mejora del {((silhouette_con_norm - silhouette_sin_norm) / abs(silhouette_sin_norm)) * 100:.1f}%")

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

scatter1 = axes[0].scatter(datos_empleados['salario_anual'], datos_empleados['edad'],
                           c=clusters_sin_norm, cmap='viridis', s=100, alpha=0.6, edgecolors='black')
axes[0].scatter(kmeans_sin_norm.cluster_centers_[:, 1], kmeans_sin_norm.cluster_centers_[:, 0],
               marker='X', s=500, c='red', edgecolors='black', linewidths=2)
axes[0].set_title(f'❌ SIN Normalización\nSilhouette: {silhouette_sin_norm:.3f}',
                 fontsize=13, fontweight='bold', color='darkred')
axes[0].set_xlabel('Salario Anual (€)')
axes[0].set_ylabel('Edad')
plt.colorbar(scatter1, ax=axes[0])

X_denorm = scaler_emp.inverse_transform(X_norm)
scatter2 = axes[1].scatter(X_denorm[:, 1], X_denorm[:, 0],
                           c=clusters_con_norm, cmap='viridis', s=100, alpha=0.6, edgecolors='black')
centroides_denorm = scaler_emp.inverse_transform(kmeans_con_norm.cluster_centers_)
axes[1].scatter(centroides_denorm[:, 1], centroides_denorm[:, 0],
               marker='X', s=500, c='red', edgecolors='black', linewidths=2)
axes[1].set_title(f'✅ CON Normalización\nSilhouette: {silhouette_con_norm:.3f}',
                 fontsize=13, fontweight='bold', color='darkgreen')
axes[1].set_xlabel('Salario Anual (€)')
axes[1].set_ylabel('Edad')
plt.colorbar(scatter2, ax=axes[1])

plt.suptitle('Impacto de la Normalización en Clustering', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Escenario 2: Modelo Predictivo Sin Outliers

**Problema:** Entrenar regresión sin detectar/eliminar outliers.

In [ ]:
# Generar datos de precios de viviendas CON outliers
np.random.seed(42)
n_casas = 200

metros_cuadrados = np.random.uniform(50, 200, n_casas)
precio_base = 2000 * metros_cuadrados + 50000
ruido = np.random.normal(0, 30000, n_casas)
precios = precio_base + ruido

# Añadir outliers (errores)
indices_outliers = [25, 67, 103, 145, 178]
precios[indices_outliers] = [50000, 750000, 80000, 850000, 60000]

df_viviendas = pd.DataFrame({
    'metros_cuadrados': metros_cuadrados,
    'precio': precios
})

print("🏠 DATASET DE VIVIENDAS")
print(f"Total: {len(df_viviendas)} viviendas")
print(f"Outliers añadidos: {len(indices_outliers)}")

X = df_viviendas[['metros_cuadrados']].values
y = df_viviendas['precio'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ❌ CON OUTLIERS
print("\n" + "="*80)
print("❌ MALA PRÁCTICA: Modelo CON Outliers")
print("="*80)

modelo_con = LinearRegression()
modelo_con.fit(X_train, y_train)
y_pred_con = modelo_con.predict(X_test)

rmse_con = np.sqrt(mean_squared_error(y_test, y_pred_con))
r2_con = r2_score(y_test, y_pred_con)
mae_con = mean_absolute_error(y_test, y_pred_con)

print(f"\nR² Score: {r2_con:.4f}")
print(f"RMSE: {rmse_con:,.2f}€")
print(f"MAE: {mae_con:,.2f}€")

# ✅ SIN OUTLIERS
print("\n" + "="*80)
print("✅ BUENA PRÁCTICA: Detectar y Eliminar Outliers")
print("="*80)

Q1 = df_viviendas['precio'].quantile(0.25)
Q3 = df_viviendas['precio'].quantile(0.75)
IQR = Q3 - Q1
outliers_mask = ((df_viviendas['precio'] < Q1 - 1.5*IQR) |
                 (df_viviendas['precio'] > Q3 + 1.5*IQR))
df_limpio = df_viviendas[~outliers_mask]

print(f"Outliers detectados: {outliers_mask.sum()}")
print(f"Datos limpios: {len(df_limpio)} ({len(df_limpio)/len(df_viviendas)*100:.1f}%)")

X_limpio = df_limpio[['metros_cuadrados']].values
y_limpio = df_limpio['precio'].values

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_limpio, y_limpio, test_size=0.2, random_state=42
)

modelo_sin = LinearRegression()
modelo_sin.fit(X_train_l, y_train_l)
y_pred_sin = modelo_sin.predict(X_test_l)

rmse_sin = np.sqrt(mean_squared_error(y_test_l, y_pred_sin))
r2_sin = r2_score(y_test_l, y_pred_sin)
mae_sin = mean_absolute_error(y_test_l, y_pred_sin)

print(f"\nR² Score: {r2_sin:.4f}")
print(f"RMSE: {rmse_sin:,.2f}€")
print(f"MAE: {mae_sin:,.2f}€")

# Comparación
print("\n" + "="*80)
print("📊 COMPARACIÓN")
print("="*80)
mejora_r2 = ((r2_sin - r2_con) / abs(r2_con)) * 100
mejora_rmse = ((rmse_con - rmse_sin) / rmse_con) * 100

comp_df = pd.DataFrame({
    'Métrica': ['R²', 'RMSE (€)', 'MAE (€)'],
    'CON Outliers': [f"{r2_con:.4f}", f"{rmse_con:,.2f}", f"{mae_con:,.2f}"],
    'SIN Outliers': [f"{r2_sin:.4f}", f"{rmse_sin:,.2f}", f"{mae_sin:,.2f}"],
    'Mejora': [f"+{mejora_r2:.1f}%", f"-{mejora_rmse:.1f}%",
               f"-{((mae_con - mae_sin) / mae_con * 100):.1f}%"]
})
display(comp_df)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CON outliers
df_viviendas['es_outlier'] = outliers_mask
axes[0].scatter(df_viviendas[~outliers_mask]['metros_cuadrados'],
                df_viviendas[~outliers_mask]['precio'],
                alpha=0.5, color='blue', s=50)
axes[0].scatter(df_viviendas[outliers_mask]['metros_cuadrados'],
                df_viviendas[outliers_mask]['precio'],
                alpha=0.8, color='red', marker='X', s=200, edgecolors='black', linewidths=2)

x_line = np.linspace(df_viviendas['metros_cuadrados'].min(),
                     df_viviendas['metros_cuadrados'].max(), 100)
y_line_con = modelo_con.predict(x_line.reshape(-1, 1))
axes[0].plot(x_line, y_line_con, 'r--', linewidth=3, label=f'R²={r2_con:.3f}')
axes[0].set_title(f'❌ CON Outliers\nRMSE: {rmse_con:,.0f}€',
                 fontsize=13, fontweight='bold', color='darkred')
axes[0].set_xlabel('Metros Cuadrados')
axes[0].set_ylabel('Precio (€)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# SIN outliers
axes[1].scatter(df_limpio['metros_cuadrados'], df_limpio['precio'],
                alpha=0.5, color='green', s=50)
y_line_sin = modelo_sin.predict(x_line.reshape(-1, 1))
axes[1].plot(x_line, y_line_sin, 'g--', linewidth=3, label=f'R²={r2_sin:.3f}')
axes[1].set_title(f'✅ SIN Outliers\nRMSE: {rmse_sin:,.0f}€',
                 fontsize=13, fontweight='bold', color='darkgreen')
axes[1].set_xlabel('Metros Cuadrados')
axes[1].set_ylabel('Precio (€)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Impacto de Outliers en Modelos de Regresión', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Impacto económico
print("\n" + "="*80)
print("💰 IMPACTO ECONÓMICO")
print("="*80)
vivienda_120m = np.array([[120]])
precio_con = modelo_con.predict(vivienda_120m)[0]
precio_sin = modelo_sin.predict(vivienda_120m)[0]

print(f"\nPredicción para vivienda de 120m²:")
print(f"   Modelo CON outliers: {precio_con:,.2f}€")
print(f"   Modelo SIN outliers: {precio_sin:,.2f}€")
print(f"   Diferencia: {abs(precio_con - precio_sin):,.2f}€")
print(f"\n⚠️  En 100 transacciones:")
print(f"   Error potencial CON outliers: {mae_con * 100:,.0f}€")
print(f"   Error reducido SIN outliers: {mae_sin * 100:,.0f}€")
print(f"   💰 AHORRO: {(mae_con - mae_sin) * 100:,.0f}€")

---

## 📊 RESUMEN EJECUTIVO

### Impacto de NO Aplicar Calidad de Datos

| Mala Práctica | Consecuencia | Impacto Económico |
|---------------|-------------|-------------------|
| No normalizar | Algoritmos sesgados | Recursos mal asignados |
| Ignorar outliers | Modelos imprecisos | 10-100x costo de corrección |
| No verificar normalidad | Correlaciones erróneas | Estrategias equivocadas |

### 💰 ROI de la Calidad de Datos

**Inversión:** 1-5€ por registro  
**Ahorro:** 10-100€ por error evitado  
**ROI:** 1,000% - 10,000%

### ✅ Conclusiones

1. **Reduce errores en 70-90%**
2. **Mejora precisión de modelos significativamente**
3. **Ahorra tiempo y dinero a largo plazo**
4. **Aumenta confianza en decisiones**
5. **Previene pérdidas económicas**

---

## 🎓 Fin del Notebook

**¡La calidad de datos es la base del éxito en Big Data e IA!**